In [18]:
import os
import cv2
import math
import random
import numpy as np
import albumentations as A
import shutil
from pathlib import Path

In [19]:
INPUT_DIR = Path("dataset_cleaned/train")
OUTPUT_DIR = Path("dataset_cleaned/train_aug")
TARGET_IMAGES = 1000
SEED = 2026
random.seed(SEED)
np.random.seed(SEED)

In [20]:
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

In [21]:
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Rotate(limit=30,p=0.7),
    A.ElasticTransform(alpha=50, sigma=5, p=0.3),
    A.RandomResizedCrop(size=(512,512),scale=(0.75,1.0),ratio=(0.9,1.1),p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.15,contrast_limit=0.15,p=0.7),
    A.RandomGamma(gamma_limit=(80,120),p=0.3),
    A.CLAHE(clip_limit=2.0,tile_grid_size=(16,16),p=0.2),
    A.GaussianBlur(blur_limit=(3,5),p=0.3),
    A.GaussNoise(std_range=(0.01,0.03),p=0.3)
],
seed=SEED
)

In [22]:
def get_image_paths(dir):
    return list(dir.rglob("*.png"))

In [23]:
def save_image(img,path):
    path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(path), img)

In [24]:
def augment(img_paths,output_path,imgs_needed):
    for i in range(imgs_needed):
        img_path = img_paths[i%len(img_paths)]
        img = cv2.imread(str(img_path),cv2.IMREAD_GRAYSCALE)
        aug=transform(image=img)["image"]
        output_name=f"{img_path.stem}_aug{i:05d}.png"
        save_image(aug, output_path / output_name)
        

In [25]:
classes = list(d for d in INPUT_DIR.iterdir() if d.is_dir())

for c in classes:
    paths = get_image_paths(c)
    imgs_needed = max(0, TARGET_IMAGES - len(paths))
    output_path = OUTPUT_DIR / c.name
    print(f"creating {imgs_needed} images for {c.name} class")
    augment(paths,output_path,imgs_needed)

creating 890 images for Biliary_Leaks class
creating 495 images for Lithiasis class
creating 803 images for Normal class
creating 745 images for Stricture class
